In [ ]:
from pathlib import Path

import torch
from rtnls_enface import FundusLoader
from rtnls_enface.qc.validators import GeneralQCValidator
from rtnls_models.inference.segmentation import (
    SegmentationEnsemble,
)
from rtnls_models.run import preprocess_for_inference
from rtnls_models.utils.config import load_config

In [ ]:
ds_path = Path("../../../samples/fundus/")
rgb_path = ds_path / "preprocessed_rgb"
ce_path = ds_path / "preprocessed_ce"
discs_path = ds_path / "discs"
vessels_path = ds_path / "vessels"

## Preprocess

In [ ]:
files = list((ds_path / "rgb").glob("*.png"))
_ = preprocess_for_inference(
    files,
    rgb_path=rgb_path,
    ce_path=ce_path,
    n_jobs=4,
)

## Disc segmentation

In [ ]:
device = torch.device("cuda:0")
disc_ensemble = SegmentationEnsemble(load_config("disc"), device=device)
disc_ensemble.predict_preprocessed(
    rgb_path, ce_path, dest_path=discs_path, num_workers=2
)

## Vessel segmentation

In [ ]:
av_ensemble = SegmentationEnsemble(load_config("vessels"), device=device)
# run the model, pass the path for the outputs
av_ensemble.predict_preprocessed(
    rgb_path, ce_path, dest_path=vessels_path, num_workers=2
)

In [ ]:
loader = FundusLoader.from_folders(discs_path, rgb_path, meta_path=ds_path / "meta.csv")

In [ ]:
validator = GeneralQCValidator()

In [ ]:
ret = loader[1]

In [ ]:
validator(ret)

In [ ]:
ret.plot()